In [49]:
import pandas as pd

metadata = pd.read_csv("HAM10000_metadata.csv")
print(metadata.head())

     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [50]:
print("Unique classes:", metadata['dx'].unique())
print("Shape:", metadata.shape)
print("Class distribution:\n", metadata['dx'].value_counts())

Unique classes: ['bkl' 'nv' 'df' 'mel' 'vasc' 'bcc' 'akiec']
Shape: (10015, 7)
Class distribution:
 dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [51]:
print("Columns:", metadata.columns.tolist())

Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']


In [52]:
print(metadata.isnull().sum())

lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64


In [53]:
# Drop 'dx_type' and 'age'
#metadata = metadata.drop(columns=['dx_type', 'age'])

# Verify
print("Remaining columns:", metadata.columns.tolist())
print(metadata.head())

Remaining columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']
     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [54]:
print(metadata.columns.tolist())

['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']


# Label Encoding 

In [55]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
metadata['label'] = le.fit_transform(metadata['dx'])

# Print mapping vertically
print("Class mapping:")
for cls, label in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls} → {label}")


Class mapping:
akiec → 0
bcc → 1
bkl → 2
df → 3
mel → 4
nv → 5
vasc → 6


# Merging the images

In [56]:
import os

# Paths to your image folders
img_dir1 = "HAM10000_images_part_1"
img_dir2 = "HAM10000_images_part_2"

def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(img_dir1, filename)
    path2 = os.path.join(img_dir2, filename)
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None  # file not found

# Apply to metadata
metadata['path'] = metadata['image_id'].apply(get_image_path)

# Re-check
print(" Paths mapped!")
print(metadata[['image_id', 'path']].head())
print("⚠️ Missing:", metadata['path'].isnull().sum())


 Paths mapped!
       image_id                                     path
0  ISIC_0027419  HAM10000_images_part_1/ISIC_0027419.jpg
1  ISIC_0025030  HAM10000_images_part_1/ISIC_0025030.jpg
2  ISIC_0026769  HAM10000_images_part_1/ISIC_0026769.jpg
3  ISIC_0025661  HAM10000_images_part_1/ISIC_0025661.jpg
4  ISIC_0031633  HAM10000_images_part_2/ISIC_0031633.jpg
⚠️ Missing: 0


# Train test split

In [57]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    metadata,
    test_size=0.2,
    stratify=metadata['label'],
    random_state=42
)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))


Training samples: 8012
Testing samples: 2003


# Image Preprocessing

In [58]:
import torchvision.transforms as transforms

# 🔹 Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),               # Step 1: Resize all images
    transforms.RandomHorizontalFlip(p=0.5),      # Step 2: Flip images (augmentation)
    transforms.RandomRotation(degrees=20),       # Step 3: Small rotations
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Step 4: Lighting variation
    transforms.ToTensor(),                       # Step 5: Convert to PyTorch tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5],   # Step 6: Normalize pixel values
                         std=[0.5, 0.5, 0.5])
])

# 🔹 Validation/Test transforms (no augmentation, only resize & normalize)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),               # Step 1: Resize
    transforms.ToTensor(),                       # Step 2: Convert to tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5],   # Step 3: Normalize
                         std=[0.5, 0.5, 0.5])
])

print(" Transform pipelines created for training and testing.")


 Transform pipelines created for training and testing.


# Create python pytorch dataset

In [59]:
from torch.utils.data import Dataset , DataLoader
from PIL import Image
import torch

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)  # reset index for safety
        self.transform = transform

        

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Get image path & label
        img_path = self.dataframe.loc[idx, 'path']
        label = self.dataframe.loc[idx, 'label']

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Apply transforms (resize, normalize, augment, etc.)
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)
    

In [60]:
# Training DataLoader
train_dataset = SkinDataset(train_df, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Testing DataLoader
test_dataset = SkinDataset(test_df, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ DataLoaders ready!")
print(f"  Training batches: {len(train_loader)}")
print(f"  Testing batches: {len(test_loader)}")


✅ DataLoaders ready!
  Training batches: 251
  Testing batches: 63


# Restnet18 CNN model

In [61]:
import torch
import torch.nn as nn
import torchvision.models as models

# Initialize ResNet18 (no weights from internet)
model = models.resnet18(weights=None)

# Load your local weights file
state_dict = torch.load("resnet18-f37072fd.pth", map_location="cpu")
model.load_state_dict(state_dict)

# Replace the final fully connected layer for 7 skin lesion classes
model.fc = nn.Linear(model.fc.in_features, 7)

print(" Pretrained ResNet18 loaded successfully!")


 Pretrained ResNet18 loaded successfully!


/var/folders/z3/b1qy845929gd42tn63d50dgm0000gn/T/ipykernel_883/655591341.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("resnet18-f37072fd.pth",

# Loss Function and Optimizer

In [62]:
import torch.nn as nn
import torch.optim as optim

# --- Loss function ---
# CrossEntropyLoss is best for multi-class classification
criterion = nn.CrossEntropyLoss()

# --- Optimizer ---
# Adam optimizer (with weight decay for regularization)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

print(" Loss function & Optimizer ready!")


 Loss function & Optimizer ready!


In [63]:
# Select device: GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print("Using device:", device)

# Move model to device
model.to(device)


Using device: mps


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Training 

In [64]:
import torch
import torch.nn as nn

# 🔹 Define number of epochs
num_epochs = 5  

for epoch in range(num_epochs):
    print(f"\n Starting Epoch {epoch+1}/{num_epochs}")

    ########################
    #   Training Phase     #
    ########################
    model.train()  # set model to training mode
    train_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track training loss & accuracy
        train_loss += loss.item()
        _, predicted = outputs.max(1)  
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        #  Print progress every 10 batches
        if (batch_idx + 1) %40 == 0:
            print(f"    Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")

    train_accuracy = 100 * correct / total

    ##########################
    #   Validation Phase     #
    ##########################
    model.eval()  # set model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():  # no gradient calculation in validation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_accuracy = 100 * val_correct / val_total

    ###########################
    #   Print Epoch Results   #
    ###########################
    print(f"Epoch [{epoch+1}/{num_epochs}] Finished | "
          f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.2f}% | "
          f"Val Loss: {val_loss/len(test_loader):.4f}, Val Acc: {val_accuracy:.2f}%")



 Starting Epoch 1/5
    Epoch 1, Batch 40/251 Loss: 0.6828
    Epoch 1, Batch 80/251 Loss: 0.5674
    Epoch 1, Batch 120/251 Loss: 0.5415
    Epoch 1, Batch 160/251 Loss: 0.6639
    Epoch 1, Batch 200/251 Loss: 0.6084
    Epoch 1, Batch 240/251 Loss: 0.4016
Epoch [1/5] Finished | Train Loss: 0.6815, Train Acc: 76.14% | Val Loss: 0.5166, Val Acc: 81.13%

 Starting Epoch 2/5
    Epoch 2, Batch 40/251 Loss: 0.4748
    Epoch 2, Batch 80/251 Loss: 0.5727
    Epoch 2, Batch 120/251 Loss: 0.4806
    Epoch 2, Batch 160/251 Loss: 0.5870
    Epoch 2, Batch 200/251 Loss: 0.5476
    Epoch 2, Batch 240/251 Loss: 0.4655
Epoch [2/5] Finished | Train Loss: 0.4786, Train Acc: 82.90% | Val Loss: 0.5046, Val Acc: 82.03%

 Starting Epoch 3/5
    Epoch 3, Batch 40/251 Loss: 0.4098
    Epoch 3, Batch 80/251 Loss: 0.3984
    Epoch 3, Batch 120/251 Loss: 0.3823
    Epoch 3, Batch 160/251 Loss: 0.3767
    Epoch 3, Batch 200/251 Loss: 0.4931
    Epoch 3, Batch 240/251 Loss: 0.3462
Epoch [3/5] Finished | Train 

# Model save

In [65]:
# Save only the learned weights
torch.save(model.state_dict(), "resnet18_skin.pth")
print(" Model weights saved successfully!")


 Model weights saved successfully!


# load model

In [66]:
'''# Select device (MPS if available, otherwise CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Recreate the same architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 7)  # 7 classes

# Load saved weights directly to device
model.load_state_dict(torch.load("resnet18_skin.pth", map_location=device))

# Move model to device
model.to(device)
model.eval()

print(f" Model loaded successfully on {device}!")'''


'# Select device (MPS if available, otherwise CPU)\ndevice = torch.device("mps" if torch.backends.mps.is_available() else "cpu")\n\n# Recreate the same architecture\nmodel = models.resnet18(weights=None)\nmodel.fc = nn.Linear(model.fc.in_features, 7)  # 7 classes\n\n# Load saved weights directly to device\nmodel.load_state_dict(torch.load("resnet18_skin.pth", map_location=device))\n\n# Move model to device\nmodel.to(device)\nmodel.eval()\n\nprint(f" Model loaded successfully on {device}!")'

In [67]:
#  Set model to evaluation mode
model.eval()

test_loss = 0.0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():  # No gradients during testing
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Track test loss
        test_loss += loss.item()

        # Get predictions
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

#  Accuracy
test_accuracy = 100 * correct / total

print("Test Results:")
print(f"  Loss: {test_loss/len(test_loader):.4f}")
print(f"  Accuracy: {test_accuracy:.2f}%")

from sklearn.metrics import classification_report




Test Results:
  Loss: 0.4752
  Accuracy: 83.67%


In [68]:
# Save only the learned weights
torch.save(model.state_dict(), "resnettest18_skin.pth")
print(" testing Model weights saved successfully!")

 testing Model weights saved successfully!


In [70]:
# Select device (MPS if available, otherwise CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Recreate the same architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 7)  # 7 classes

# Load saved weights directly to device
model.load_state_dict(torch.load("resnet18_skin.pth", map_location=device))

# Move model to device
model.to(device)
model.eval()

print(f" Model loaded successfully on {device}!")

 Model loaded successfully on mps!


/var/folders/z3/b1qy845929gd42tn63d50dgm0000gn/T/ipykernel_883/449887553.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("resnet18_skin.

# Classification Report 

In [71]:
# 🔹 Ensure model is in evaluation mode
model.eval()
from sklearn.metrics import classification_report
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

#  Generate classification report
print("📊 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=[
    "akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"   # <-- replace with your class names
]))


📊 Classification Report:
              precision    recall  f1-score   support

       akiec       0.96      0.38      0.55        65
         bcc       0.70      0.83      0.76       103
         bkl       0.70      0.76      0.73       220
          df       0.79      0.65      0.71        23
         mel       0.56      0.63      0.59       223
          nv       0.92      0.91      0.92      1341
        vasc       0.95      0.75      0.84        28

    accuracy                           0.84      2003
   macro avg       0.80      0.70      0.73      2003
weighted avg       0.85      0.84      0.84      2003



# Testing single image

In [8]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# 1. Recreate the same architecture you used in training
model = models.resnet18(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)   # put your number of classes here

# 2. Load the saved weights (state_dict)
state_dict = torch.load("resnet18_skin.pth", map_location=torch.device('mps'))
model.load_state_dict(state_dict)
model.eval()   # ✅ now works because it's a real model

# 3. Define preprocessing (must match training pipeline)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  
])

# 4. Load and preprocess a single test image
img_path = "images-4.jpeg"
img = Image.open(img_path).convert("RGB")
img = transform(img).unsqueeze(0)  # add batch dimension

# 5. Inference
with torch.no_grad():
    outputs = model(img)
    _, predicted = torch.max(outputs, 1)

# 6. Map prediction to class name
class_names = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]  # adjust if different
print("Predicted class:", class_names[predicted.item()])
print("Raw outputs:", outputs)


Predicted class: nv
Raw outputs: tensor([[-2.1912, -2.6029,  0.9746, -2.5153, -0.5657,  1.7167, -1.3952]])


/var/folders/z3/b1qy845929gd42tn63d50dgm0000gn/T/ipykernel_3994/2290837464.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("resnet18_skin.pth", 